# YaRN vs native RoPE on needle-in-a-haystackFree-tier recipe (Kaggle 2xT4 / Colab T4). The harness lives in this repo; thisnotebook only wires it up. Every run appends to a JSONL, so a killed sessionresumes instead of restarting.**Run order for a session you may lose:** install -> smoke -> base -> yarn -> analyze.

In [ ]:
# 1) Get the code (upload the folder as a Kaggle dataset, or clone a repo / use Drive)import os, sys, pathlibREPO = os.environ.get("NIAH_REPO", "/kaggle/working/yarn-haystack")if not pathlib.Path(REPO, "run.py").exists():    raise SystemExit(f"!! put run.py/niah.py/analyze.py in {REPO} first")os.chdir(REPO); sys.path.insert(0, REPO)WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"print("repo:", REPO, "| work:", WORK)

In [ ]:
# 2) Install ONLY what is missing or genuinely too old.##    Why not `pip install -U` everything: Kaggle/Colab preinstall a CUDA-matched#    torch, matplotlib, datasets and accelerate. Upgrading them buys nothing here#    and provokes conflicts with preinstalled tools (ydata-profiling pins#    matplotlib<=3.10; bigframes is missing google-cloud-bigquery-storage). Those#    conflicts are reported by pip after ANY install, because pip re-checks the#    whole environment, and they are harmless: we import neither package.##    The ONE hard requirement is transformers>=4.51 - below that, transformers#    silently drops original_max_position_embeddings from rope_scaling and the#    "yarn" arm stops being YaRN. That is asserted below, not assumed.import importlib.metadata as mdimport subprocess, sysdef version(pkg):    try:        return md.version(pkg)    except md.PackageNotFoundError:        return NoneTF_MIN = (4, 51)def too_old(v, floor):    return v is None or tuple(int(x) for x in v.split(".")[:2]) < floorto_install = []if too_old(version("transformers"), TF_MIN):    to_install.append("transformers>=4.51")for pkg in ("accelerate", "datasets", "bitsandbytes"):   # bitsandbytes: only for --load-in-4bit    if version(pkg) is None:        to_install.append(pkg)if to_install:    print("pip install:", " ".join(to_install), flush=True)    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *to_install], check=True)else:    print("nothing to install - using what the image ships")print()for pkg in ("transformers", "torch", "accelerate", "datasets", "bitsandbytes", "matplotlib"):    print(f"  {pkg:14s} {version(pkg)}")import torchprint(f"\ntorch {torch.__version__} | cuda available: {torch.cuda.is_available()} | gpus: {torch.cuda.device_count()}")for i in range(torch.cuda.device_count()):    p = torch.cuda.get_device_properties(i)    print(f"  gpu{i}: {p.name}, {p.total_memory / 2**30:.1f} GiB")# Hard gate. Fail here, in 5 seconds, rather than after a 9-hour session on a# treatment arm that is secretly a copy of the control._v = version("transformers")assert _v and tuple(int(x) for x in _v.split(".")[:2]) >= TF_MIN, (    f"need transformers>=4.51, have {_v}. Without it, original_max_position_embeddings "    "inside rope_scaling is ignored and the YaRN arm is not YaRN.")print("\ntransformers >= 4.51: OK")

In [ ]:
# 3) THE EXPERIMENT CONFIG. Change these four lines, nothing else.##    Wave 1 (one T4, cheapest path to a number):#        MODEL = "Qwen/Qwen2.5-7B-Instruct"   # native 32k, 56 KiB KV/token#    Wave 2 (needs Kaggle 2xT4 + vLLM TP=2: Qwen3 KV is 144 KiB/token => 18 GiB at 128k):#        MODEL = "Qwen/Qwen3-8B"              # native 32k (config says 40960), 144 KiB/token##    Do NOT use Qwen3.5-*, Qwen3-*-Instruct-2507, Qwen3-Next-* or Qwen3-Coder-*:#    they are natively 256k, so at <=128k there is no extrapolation failure for#    YaRN to fix. run.py blocks them for you.MODEL       = "Qwen/Qwen2.5-7B-Instruct"ORIG_MAX    = 32768            # the TRAINED prompt window, NOT config.json's 40960FACTOR      = 4.0              # -> scaled range 131072; agent guidance: match your target, 2.0 for 64kLENGTHS     = "4096,8192,16384,32768,65536,131072"DEPTHS      = "0.0,0.1,0.25,0.5,0.75,0.9,1.0"EXTRA       = ["--load-in-4bit", "--attn", "sdpa"]   # identical for BOTH arms, alwaysTAG         = MODEL.split("/")[-1].lower().replace(".", "") + f"_f{FACTOR:g}"print(TAG, LENGTHS)

In [ ]:
# 4) PREFLIGHT - three checks that cost seconds and catch the failures which#    otherwise show up later as a confusing CalledProcessError.import os, urllib.request# 4a) GPU. Kaggle gives 2xT4, Colab 1xT4; everything downstream depends on it.import torchprint(f"torch {torch.__version__} | cuda available: {torch.cuda.is_available()} | gpus: {torch.cuda.device_count()}")for i in range(torch.cuda.device_count()):    p = torch.cuda.get_device_properties(i)    print(f"  gpu{i}: {p.name}, {p.total_memory / 2**30:.1f} GiB")if not torch.cuda.is_available():    print("!! NO CUDA. Kaggle: Settings -> Accelerator -> GPU T4 x2 | Colab: Runtime -> Change runtime type")    print("!! run.py will fall back to float32 on CPU: fine for a correctness smoke test,")    print("!! useless for 64k/128k. Do not draw conclusions from CPU timings.")# 4b) Internet. OFF by default in Kaggle notebooks, and both the tokenizer and the#     corpus need it. This is the most common reason run.py dies in the first second.try:    print("huggingface.co ->", urllib.request.urlopen("https://huggingface.co", timeout=15).status)except Exception as exc:    raise SystemExit(f"no internet: {exc!r}\nKaggle: Settings -> Internet -> On (and accept the terms).")# 4c) Corpus, in isolation from the model: separates "download problem" from#     "harness problem". Downloads ~4 MB once, then reuses ./cache.import niah_text = niah.load_corpus(None, os.path.join(REPO, "cache"))print(f"corpus OK: {len(_text)/1e6:.1f}M chars, cached under ./cache")

In [ ]:
import subprocess, sys, os, collectionsRESULTS = f"{WORK}/results"; os.makedirs(RESULTS, exist_ok=True)def niah(*args):    """Run the harness as a subprocess, streaming its output live.    -u is not decoration: without it the child's stdout is block-buffered through    a pipe, so a 40-minute 128k case looks frozen. On failure the last 40 lines    are put INTO the exception, because a bare CalledProcessError hides the    actual error - which is the one thing you need to read.    """    cmd = [sys.executable, "-u", "run.py", *args]    print("$", " ".join(cmd), flush=True)    tail = collections.deque(maxlen=40)    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,                          text=True, bufsize=1) as p:        for line in p.stdout:            print(line, end="", flush=True)            tail.append(line.rstrip())        p.wait()    if p.returncode:        raise RuntimeError(f"run.py exited with {p.returncode}. Last lines of its output:\n  "                           + "\n  ".join(tail))# 5) SMOKE TEST FIRST: tiny model, one length.#    run.py raises if rope_scaling did not stick, but read the logged config yourself.niah("--model", "Qwen/Qwen2.5-1.5B-Instruct", "--arm", "yarn", "--factor", "4",     "--original-max", "32768", "--lengths", "4096", "--depths", "0.5",     "--max-new-tokens", "24", "--out", f"{RESULTS}/smoke.jsonl")

In [ ]:
import subprocess, sys, os, collectionsRESULTS = f"{WORK}/results"; os.makedirs(RESULTS, exist_ok=True)def niah(*args):    """Run the harness as a subprocess, streaming its output live.    -u is not decoration: without it the child's stdout is block-buffered through    a pipe, so a 40-minute 128k case looks frozen. On failure the last 40 lines    are put INTO the exception, because a bare CalledProcessError hides the    actual error - which is the one thing you need to read.    """    cmd = [sys.executable, "-u", "run.py", *args]    print("$", " ".join(cmd), flush=True)    tail = collections.deque(maxlen=40)    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,                          text=True, bufsize=1) as p:        for line in p.stdout:            print(line, end="", flush=True)            tail.append(line.rstrip())        p.wait()    if p.returncode:        raise RuntimeError(f"run.py exited with {p.returncode}. Last lines of its output:\n  "                           + "\n  ".join(tail))# 6) ARM 1 - native RoPE, extrapolated past its window. Expect collapse at 64k/128k.#    This is not a throwaway arm: yarn@128k - base@128k IS the YaRN effect.niah("--model", MODEL, "--arm", "base",     "--lengths", LENGTHS, "--depths", DEPTHS, "--max-new-tokens", "64",     "--out", f"{RESULTS}/base_{TAG}.jsonl", *EXTRA)

In [ ]:
import subprocess, sys, os, collectionsRESULTS = f"{WORK}/results"; os.makedirs(RESULTS, exist_ok=True)def niah(*args):    """Run the harness as a subprocess, streaming its output live.    -u is not decoration: without it the child's stdout is block-buffered through    a pipe, so a 40-minute 128k case looks frozen. On failure the last 40 lines    are put INTO the exception, because a bare CalledProcessError hides the    actual error - which is the one thing you need to read.    """    cmd = [sys.executable, "-u", "run.py", *args]    print("$", " ".join(cmd), flush=True)    tail = collections.deque(maxlen=40)    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,                          text=True, bufsize=1) as p:        for line in p.stdout:            print(line, end="", flush=True)            tail.append(line.rstrip())        p.wait()    if p.returncode:        raise RuntimeError(f"run.py exited with {p.returncode}. Last lines of its output:\n  "                           + "\n  ".join(tail))# 7) ARM 2 - same weights, YaRN. Byte-identical setup otherwise: same EXTRA, same grid.niah("--model", MODEL, "--arm", "yarn", "--factor", str(FACTOR),     "--original-max", str(ORIG_MAX),     "--lengths", LENGTHS, "--depths", DEPTHS, "--max-new-tokens", "64",     "--out", f"{RESULTS}/yarn_{TAG}.jsonl", *EXTRA)

In [ ]:
# 8) Report: summary.md + heatmaps.png + CSVs, incl. the delta-vs-base table.import subprocess, sys, globres = sorted(glob.glob(f"{RESULTS}/base_{TAG}.jsonl") + glob.glob(f"{RESULTS}/yarn_{TAG}.jsonl"))subprocess.run([sys.executable, "analyze.py", *res, "--outdir", f"{WORK}/report_{TAG}"], check=True)from IPython.display import Image, Markdown, displaydisplay(Markdown(open(f"{WORK}/report_{TAG}/summary.md").read()))display(Image(filename=f"{WORK}/report_{TAG}/heatmaps.png"))

### Reading the result- `base` at 64k/128k collapsing to ~0 and `yarn` holding up = YaRN doing its job.- Check the **4k/8k/16k row**: Qwen's own model card warns that static YaRN "may  potentially degrade model performance" on short texts. A drop there is real and  is the price of the 128k gain, not measurement noise.- Depth matters: NIAH tends to break in the middle ("lost in the middle") before  it breaks at the edges. A uniform-looking heatmap row at every length is worth  more than a single number.### If 7B @ 128k OOMs`--load-in-4bit --attn sdpa` needs ~11 GiB (NF4 weights + 7 GiB KV). If it still OOMs:- `Qwen/Qwen2.5-3B-Instruct` in fp16 (~10.3 GiB at 128k): same protocol, weaker model;- `--device-map auto` to shard across Kaggle's 2xT4 (slow, but it fits);- `--backend vllm --tensor-parallel-size 2` (fastest, iff vLLM installs cleanly).Never mix these flags across arms — a 4-bit `base` vs fp16 `yarn` measures quantization.### Persisting across sessions`/kaggle/working` survives a session stop. The JSONL is append-only and fsync'd,so `run.py` skips finished cases automatically — just re-run the same cell. Syncwith `--upload <user>/yarn-niah` for belt and braces.